# Regressions export (Folowup study)
Carolina Carreira, McKenna McCall


# Setup

In [1]:
import pandas as pd
import seaborn as sns
#import matplotlib.pylab as plt
import surveyfieldsfollowup
#import numpy as np

pd.set_option('display.width', 400)
pd.set_option('display.max_columns', 10)

sns.set_theme()

ENCODING = 'utf-8-sig'
RESPONSE_ID = "ResponseId"
IMPORT_DIR = "./export_followup/"
EXPORT_DIR = "./data/followup/"
PATH_EXCLUSION = EXPORT_DIR+"exclusions.csv"
PATH_DEMOGRAPHIC_QUAL = EXPORT_DIR+"demographic-recoded.csv"
PATH_WILLINGNESS = EXPORT_DIR+"willingness-recoded.csv"
PATH_QUALITATIVE = EXPORT_DIR+"qualitative_data.csv"

/var/folders/tv/m58r9qz162g9dn43wdl5jk7w0000gq/T/ipykernel_33398/3537373160.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### Read data and some basic cleaning

* Read scored data
* Re-label data

In [2]:
results = pd.read_csv(IMPORT_DIR+'anonymized_data.csv', encoding=ENCODING)
results.columns = surveyfieldsfollowup.allFields

### Recode fields for regressions

This block recodes several of the background/experience fields into more
condensed versions more suitable for running regression analyses.
* Reduce number of levels in age, education, and gender
* Convert demographic specific things to True and False booleans (e.g. IoT device ownsership)

In [3]:
def aux_apply_map_clean(df, map, column):
  # applies map and keeps the column name
  df[column+'aux'] = df[column].apply(lambda x: map[x])
  df[column] = df[column+"aux"].copy()
  df = df.drop(column+"aux", axis=1)
  return df


def recode_for_regression(df):
  ageMap = {
    '18-24 years old': '18-24',
    '25-34 years old': '25-34',
    '35-44 years old': '35-44',
    '45-54 years old': '45+',
    '55-64 years old': '45+',
    '65+ years old': '45+',
    'Prefer not to say': 'Prefer not to say'
  }
  
  df['age_reduced'] = df['age'].apply(lambda age: ageMap[age])
  df["age"] = df["age_reduced"].copy()
  df = df.drop('age_reduced', axis=1)
  

  # Recode education into two buckets
  educationMap = {
    'Associates or technical degree': 'Bachelor or Graduate degree',
    'Bachelor’s degree': 'Bachelor or Graduate degree',
    'Graduate or professional degree (MA, MS, MBA, PhD, JD, MD, DDS etc.)': 'Bachelor or Graduate degree',
    'High school diploma or GED': 'High school or below',
    'Some high school or less': 'High school or below',
    'Other': 'Other',
    'Some college, but no degree': 'High school or below',
    'Prefer not to say': 'Prefer not to say'
  } 
  df['education_reduced'] = df['education'].apply(lambda edu: educationMap[edu])
  df["education"] = df["education_reduced"].copy()
  df = df.drop("education_reduced", axis=1) 

  

  # Recode Medical Job to True or False
  medicalExpMap = {
    'Yes; as a participant': True,
    'Yes; as a researcher': True,
    'Yes, as a participant': True, # follow-up only
    'Yes, as a researcher': True, # follow-up only
    'No': False,
    'I’m not sure/Other': False
  }
  df['previous_medical_research'] = df['demo-medical'].apply(lambda x: medicalExpMap[x])
  
  # Recode Medical Experience to True or False
  medicalJobMap = {
    'Yes': True,
    'No, but I have been in the past': True,
    'No': False,
    'I’m not sure/Other': False
  }
  df['medical_job'] = df['demo-medical-employ'].apply(lambda x: medicalJobMap[x])
  

  # Recode ownership of IoT to True or False
  IoTMap = {
    'Yes, and I use the smart features': True,
    'Yes, but I don’t use the smart features': True,
    'No': False,
    "No, but I have in the past": True,
    'I’m not sure/Other': False
  }
  df['IoT_ownership'] = df['demo-iot'].apply(lambda x: IoTMap[x])

  
  FAQ_rebinned = {
    'No FAQ': False,
    'Show': True,
    'Hide': True
  }
  df['FAQ_rebinned'] = df['FAQ'].apply(lambda x: FAQ_rebinned[x])

  EXPLN_rebinned = {
    'No explanation': False,
    'Hardware': True,
    'Unsubstantial': True,
    'Trust': True
  }
  df['EXPLN_rebinned'] = df['EXPLN'].apply(lambda x: EXPLN_rebinned[x])

  # Recode IoT automation to True or False
  IoTMap = {
    'Yes, I currently use home automation': True,
    'Yes, I have used home automation, but not currently': True,
    'No, I don’t own smart devices': False,
    "No, I do not use home automation, but I do own smart devices": False,
    'I’m not sure/Other': False
  }
  df['IoT_automation'] = df['demo-tap'].apply(lambda x: IoTMap[x])

  # Recode employment in IT field
  TechMap = {
    "Yes": True,
    "No": False,
    "No, but I have been in the past": True,
    "I’m not sure/Other": False
  }
  df['CS_Employment'] = df['demo-cs-employ'].apply(lambda x: TechMap[x])

  # Recode education in CS 
  TechMap = {
    "Yes": True,
    "No": False,
    "I'm not sure/Other": False
  }
  df['CS_Education'] = df['demo-cs-education'].apply(lambda x: TechMap[x])

  # willingness = {
  #   'Definitely would': True,
  #   'Maybe would': True,
  #   'Would not ': False,
  #   'Not sure (Why?)': False
  # }
  willingness = {
    'Definitely would': 'Definitely would',
    'Maybe would': 'Maybe would',
    'Would not ': 'Would not',
    'Not sure (Why?)': 'Would not'
  }
  df = aux_apply_map_clean(df, willingness, "willingness-medical")
  df = aux_apply_map_clean(df, willingness, "willingness-iot")

   # Recode safe
  # safe = {
  #   "Completely safe": False,
  #   "Mostly safe": False,
  #   "Somewhat safe": False,
  #   "Not at all safe": True
  # }
  safe = {
    "Completely safe": "Completely safe",
    "Mostly safe": "Completely safe",
    "Somewhat safe": "Somewhat safe",
    "Not at all safe": "Not at all safe"
  }
  df = aux_apply_map_clean(df, safe, "safety-medical-overall-init")
  df = aux_apply_map_clean(df, safe, "safety-iot-overall-init")
  df = aux_apply_map_clean(df, safe, "safety-medical-overall-post")
  df = aux_apply_map_clean(df, safe, "safety-iot-overall-post")

  # safe_long_likert = {
  #   "Definitely safe": False,
  #   "Somewhat safe": False,
  #   "Neither safe nor unsafe": False,
  #   "Somewhat unsafe": True,
  #   "Definitely unsafe": True,
  # }

  safe_long_likert = {
    "Definitely safe": "Safe",
    "Somewhat safe": "Safe",
    "Neither safe nor unsafe": "Neither safe nor unsafe",
    "Somewhat unsafe": "Unsafe",
    "Definitely unsafe": "Unsafe"
  }
  # iot
  # df = aux_apply_map_clean(df, safe_long_likert, "safety-medical-tee")
  # df = aux_apply_map_clean(df, safe_long_likert, "safety-medical-hospital")
  # df = aux_apply_map_clean(df, safe_long_likert, "safety-medical-people")
  # df = aux_apply_map_clean(df, safe_long_likert, "safety-medical-data")
  # df = aux_apply_map_clean(df, safe_long_likert, "safety-medical-purpose")


  # # medical
  # df = aux_apply_map_clean(df, safe_long_likert, "safety-iot-tee")
  # df = aux_apply_map_clean(df, safe_long_likert, "safety-iot-company")
  # df = aux_apply_map_clean(df, safe_long_likert, "safety-iot-people")
  # df = aux_apply_map_clean(df, safe_long_likert, "safety-iot-data")
  # df = aux_apply_map_clean(df, safe_long_likert, "safety-iot-purpose")
  

  df['gender'] = df['gender'].replace({
    'Female': 'Female NB or SD',
    'Prefer to self-describe': 'Female NB or SD',
    'Non-binary': 'Female NB or SD',
    "Non-binary / third gender": 'Female NB or SD'
})  
  scenario = {
    "Medical-Simple": "MedicalSimple",
    "Medical-Complex": "MedicalComplex",
    "IoT-Simple": "IoTSimple",
    "IoT-Complex": "IoTComplex"
  }
  
  df = aux_apply_map_clean(df, scenario, "SCENARIO_Med")
  df = aux_apply_map_clean(df, scenario, "SCENARIO_IoT")
  return df

results['EXPLN'].fillna('No explanation', inplace=True)
results['FAQ'].fillna('No FAQ', inplace=True)
results_recoded = recode_for_regression(results.copy())




  

/var/folders/tv/m58r9qz162g9dn43wdl5jk7w0000gq/T/ipykernel_33398/3729069916.py:197: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  results['EXPLN'].fillna('No explanation', inplace=True)
/var/folders/tv/m58r9qz162g9dn43wdl5jk7w0000gq/T/ipykernel_33398/3729069916.py:198: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting 

In [4]:
variables_to_melt = {
    "willingness": ["willingness-medical","willingness-iot"],
    "safety_overall_init": ["safety-medical-overall-init","safety-iot-overall-init"],
    "safety_tee": ["safety-medical-tee","safety-iot-tee"],
    "safety_companyHospital": ["safety-medical-hospital","safety-iot-company"],
    "safety_people": ["safety-medical-people","safety-iot-people"],
    "safety_data": ["safety-medical-data","safety-iot-data"],
    "safety_purpose": ["safety-medical-purpose","safety-iot-purpose"],
    "safety_overall_post": ["safety-medical-overall-post","safety-iot-overall-post"]
}

def melt_aux(df, final_df, new_colum, original_columns):
    # for comprehension medical
    med_df = pd.melt(df, id_vars=['id', 'SCENARIO_Med'], value_vars=[original_columns[0]],
                        var_name='Question', value_name=new_colum)
    med_df['scenario'] = med_df['SCENARIO_Med']
    med_df.drop(columns=['Question', 'SCENARIO_Med'], inplace=True)

    # for comprehension iot
    iot_df = pd.melt(df, id_vars=['id', 'SCENARIO_IoT'], value_vars=[original_columns[1]],
                        var_name='Question', value_name=new_colum)
    iot_df['scenario'] = iot_df['SCENARIO_IoT']
    iot_df.drop(columns=['Question', 'SCENARIO_IoT'], inplace=True)

    current_df = pd.concat([med_df, iot_df], ignore_index=True)
    final_df = pd.merge(final_df, current_df, on=['id', 'scenario'], how='outer')
    return final_df


def aggregate_scenario_comprehension(df):
    final_df = pd.DataFrame()
    for i in range(1, 13):
        m_col = f'M{i}'
        i_col = f'I{i}'
        q_col = f'Q{i}'
        
        # for comprehension medical
        med_df = pd.melt(df, id_vars=['id', 'SCENARIO_Med'], value_vars=[m_col],
                         var_name='Question', value_name=q_col)
        med_df['scenario'] = med_df['SCENARIO_Med']
        med_df.drop(columns=['Question', 'SCENARIO_Med'], inplace=True)

        # for comprehension iot
        iot_df = pd.melt(df, id_vars=['id', 'SCENARIO_IoT'], value_vars=[i_col],
                         var_name='Question', value_name=q_col)
        iot_df['scenario'] = iot_df['SCENARIO_IoT']
        iot_df.drop(columns=['Question', 'SCENARIO_IoT'], inplace=True)

        current_df = pd.concat([med_df, iot_df], ignore_index=True)

        if final_df.empty:
            final_df = current_df
        else:
            final_df = pd.merge(final_df, current_df, on=['id', 'scenario'], how='outer')

    for new_column_name in variables_to_melt:
        final_df = melt_aux(df, final_df,new_column_name, variables_to_melt[new_column_name])

    final_df = pd.merge(final_df, current_df, on=['id', 'scenario'], how='outer')

    final_df.sort_values('id', inplace=True)
    final_df.reset_index(drop=True, inplace=True)

    return final_df

This code is to merge the demographic predictors for the regression to fit better (hopefully)

In [5]:
def reduce_predictors(df):
  df['combined_IoT_Experience'] = df['IoT_ownership'] | df['IoT_automation']
  df['combined_Medical_Experience'] = df['previous_medical_research'] | df['medical_job']
  df['combined_CS_Experience'] = df['CS_Employment'] | df['CS_Education']

  return df

def augment_scenarios(df):
  
  scenario_complexity = {
    "MedicalSimple": "Simple",
    "MedicalComplex": "Complex",
    "IoTSimple": "Simple",
    "IoTComplex": "Complex"
  }

  scenario_scenario = {
    "MedicalSimple": "Medical",
    "MedicalComplex": "Medical",
    "IoTSimple": "IoT",
    "IoTComplex": "IoT"
  }

  df['scenario_complexity'] = df['scenario'].apply(lambda x: scenario_complexity[x])
  df['scenario_scenario'] = df['scenario'].apply(lambda x: scenario_scenario[x])
  return df

In [6]:
# this is for tests
data_example = {
    'id': [1, 2],
    'M1': [5, 3], 'M2': [7, 2], 'M3': [9, 1], 'M4': [2, 4], 'M5': [8, 6], 'M6': [3, 7], 'M7': [4, 5], 'M8': [6, 8], 'M9': [5, 9], 'M10': [7, 3], 'M11': [1, 4], 'M12': [8, 5],
    'I1': [4, 6], 'I2': [6, 4], 'I3': [5, 3], 'I4': [7, 2], 'I5': [3, 8], 'I6': [2, 7], 'I7': [8, 1], 'I8': [1, 9], 'I9': [9, 5], 'I10': [6, 3], 'I11': [2, 8], 'I12': [4, 1],
    'SCENARIO_Med': ['Medical-Simple', 'Medical-Complex'],
    'SCENARIO_IoT': ['IoT-Complex', 'IoT-Simple']
}
df = pd.DataFrame(data_example)

comprehension_aggregated = aggregate_scenario_comprehension(results_recoded)

# merge comprehnsion back into the rest of the data
predictors = ["EXPLN", "FAQ", "age", "gender", "IoT_ownership", "IoT_automation", "CS_Employment", "CS_Education", "previous_medical_research", "medical_job", "education", "FAQ_rebinned", "EXPLN_rebinned"]


results_recoded_compre = results_recoded[predictors+["id"]].copy()
export_compre = pd.merge(comprehension_aggregated, results_recoded, on='id', how='left')
export_compre = reduce_predictors(export_compre)

export_all = pd.merge(comprehension_aggregated, results_recoded, on='id', how='left')
export_all = reduce_predictors(export_all)

export_all = augment_scenarios(export_all)
print(export_all.columns)
export_all.to_csv(EXPORT_DIR+'scores_all_levels_safety.csv', index=False)



Index(['id', 'Q1', 'scenario', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8',
       ...
       'FAQ_rebinned', 'EXPLN_rebinned', 'IoT_automation', 'CS_Employment', 'CS_Education', 'combined_IoT_Experience', 'combined_Medical_Experience', 'combined_CS_Experience', 'scenario_complexity', 'scenario_scenario'], dtype='object', length=148)
